In [ ]:
from google.colab import files
uploaded = files.upload()
from scipy.io import arff
import pandas as pd

# Chargement du fichier ARFF
data = arff.loadarff("Training Dataset.arff")

Saving Training Dataset.arff to Training Dataset.arff


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, learning_curve
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.metrics import classification_report, roc_curve, auc, accuracy_score, precision_recall_curve
from sklearn.metrics import confusion_matrix, average_precision_score, f1_score
from sklearn.feature_selection import RFECV, mutual_info_classif

from sklearn.ensemble import IsolationForest
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline

# Set display precision & style
np.set_printoptions(precision=4)
pd.set_option('display.precision', 4)
sns.set_style('whitegrid')
plt.style.use('seaborn-v0_8-pastel')

# ---------------------------
# 0. Chargement du dataset ARFF
# ---------------------------
print("="*80)
print("PHISHING DETECTION AVEC XGBOOST - ANALYSE AVANCÉE")
print("="*80)

# Fonction pour charger un fichier ARFF en DataFrame pandas
def load_arff(file_path):
    data = []
    attributes = []
    data_section_started = False

    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()

            # Ignorer les lignes vides et les commentaires
            if not line or line.startswith('%'):
                continue

            # Collecter les attributs
            if line.lower().startswith('@attribute'):
                attr_name = line.split()[1].strip()
                attributes.append(attr_name)

            # Détecter le début de la section de données
            elif line.lower().startswith('@data'):
                data_section_started = True

            # Collecter les données
            elif data_section_started:
                values = line.split(',')
                data.append(values)

    # Créer le DataFrame
    df = pd.DataFrame(data, columns=attributes)

    # Convertir les valeurs en numérique (en ignorant les erreurs pour les valeurs non-numériques)
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='ignore')

    return df

# Charger le dataset (vous devrez spécifier le chemin vers votre fichier ARFF)
file_path = 'Training Dataset.arff'  # Modifiez ceci
df = load_arff(file_path)

# Identifier la colonne cible (supposons que c'est la dernière colonne)
target_column = df.columns[-1]

# 1. Vérifier les valeurs uniques dans la colonne cible
print("Valeurs uniques initiales dans la colonne cible:", np.unique(df[target_column]))

# 2. Transformer la colonne cible en classification binaire (0 et 1)
df[target_column] = np.where(df[target_column] > 0, 1, 0)

# 3. Vérifier après transformation
print("Valeurs après transformation:", np.unique(df[target_column]))

print(f"Dimensions du dataset: {df.shape[0]} lignes × {df.shape[1]} colonnes")

# Afin de vous assurer que les valeurs numériques sont correctement interprétées
df = df.astype(float)

# ---------------------------
# 1. Analyse exploratoire des données
# ---------------------------
print("\n" + "="*50)
print("ANALYSE EXPLORATOIRE DES DONNÉES")
print("="*50)

# Statistiques descriptives
print("\nStatistiques descriptives:")
print(df.describe())

# Vérification des valeurs manquantes
print("\nValeurs manquantes par colonne:")
print(df.isnull().sum())

# Distribution de la variable cible
print("\nDistribution de la variable cible:")
target_counts = df[target_column].value_counts()
print(target_counts)
print(f"Pourcentage de classe positive: {target_counts[1] / len(df) * 100:.2f}%")
print(f"Pourcentage de classe négative: {target_counts[0] / len(df) * 100:.2f}%")

# Visualisation de la distribution de la cible
plt.figure(figsize=(8, 6))
sns.countplot(x=df[target_column])
plt.title('Distribution de la variable cible')
plt.xlabel('Classe')
plt.ylabel('Nombre d\'échantillons')
plt.savefig('target_distribution.png')
plt.close()

# ---------------------------
# 2. Préparation des données
# ---------------------------
print("\n" + "="*50)
print("PRÉPARATION DES DONNÉES")
print("="*50)

# Séparation des caractéristiques et de la cible
X = df.drop(target_column, axis=1)
y = df[target_column]

# Division train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Taille de l'ensemble d'entraînement: {X_train.shape[0]} échantillons")
print(f"Taille de l'ensemble de test: {X_test.shape[0]} échantillons")

# Normalisation des données
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Détection et gestion des valeurs aberrantes
print("\nDétection des valeurs aberrantes avec Isolation Forest...")
iso_forest = IsolationForest(contamination=0.05, random_state=42)
outliers = iso_forest.fit_predict(X_train_scaled)
print(f"Nombre de valeurs aberrantes détectées: {(outliers == -1).sum()}")

# Gestion du déséquilibre des classes
print("\nGestion du déséquilibre des classes avec SMOTE...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
print(f"Distribution après rééchantillonnage: {np.bincount(y_train_resampled.astype(int))}")

# Sélection des caractéristiques
print("\nSélection récursive des caractéristiques avec validation croisée...")
xgb_base = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
rfecv = RFECV(estimator=xgb_base, step=1, cv=StratifiedKFold(5), scoring='f1', n_jobs=-1)
rfecv.fit(X_train_resampled, y_train_resampled)

print(f"Nombre optimal de caractéristiques: {rfecv.n_features_}")
print(f"Caractéristiques sélectionnées: {X.columns[rfecv.support_].tolist()}")

# Utiliser uniquement les caractéristiques sélectionnées
X_train_selected = X_train_resampled[:, rfecv.support_]
X_test_selected = X_test_scaled[:, rfecv.support_]

# ---------------------------
# 3. Modélisation avec XGBoost
# ---------------------------
print("\n" + "="*50)
print("MODÉLISATION AVEC XGBOOST")
print("="*50)

# Création du modèle XGBoost
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    booster='gbtree',
    learning_rate=0.1,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    n_estimators=100,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Grille de paramètres pour l'optimisation
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'n_estimators': [50, 100, 200]
}

# Optimisation des hyperparamètres avec validation croisée
print("\nOptimisation des hyperparamètres...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train_selected, y_train_resampled)

print(f"Meilleurs paramètres: {grid_search.best_params_}")
print(f"Meilleure performance (F1): {grid_search.best_score_:.4f}")

# Modèle final avec les meilleurs paramètres
# Modèle final avec les meilleurs paramètres
best_xgb = grid_search.best_estimator_

# Entraînement simple sans paramètres supplémentaires
best_xgb.fit(X_train_selected, y_train_resampled)

# Si vous voulez quand même avoir des informations sur les performances sur l'ensemble de test
y_test_pred = best_xgb.predict(X_test_selected)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Précision sur l'ensemble de test: {test_accuracy:.4f}")

# Importance des caractéristiques
print("\nImportance des caractéristiques:")
feature_names = X.columns[rfecv.support_].tolist()
importance = best_xgb.feature_importances_
indices = np.argsort(importance)[::-1]

for i, idx in enumerate(indices):
    if i < 10:  # Afficher les 10 caractéristiques les plus importantes
        print(f"{feature_names[idx]}: {importance[idx]:.4f}")

# Visualisation de l'importance des caractéristiques
plt.figure(figsize=(12, 8))
xgb.plot_importance(best_xgb, max_num_features=10, height=0.8)
plt.title('Importance des caractéristiques (XGBoost)')
plt.savefig('feature_importance_xgb.png')
plt.close()

# Courbes d'apprentissage
print("\nGénération des courbes d'apprentissage...")
train_sizes, train_scores, test_scores = learning_curve(
    best_xgb, X_train_selected, y_train_resampled, cv=5,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='f1', n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color='r', label='Score d\'entraînement')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='r')
plt.plot(train_sizes, test_mean, 'o-', color='g', label='Score de validation croisée')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color='g')
plt.xlabel('Nombre d\'échantillons d\'entraînement')
plt.ylabel('Score F1')
plt.title('Courbes d\'apprentissage pour XGBoost')
plt.legend(loc='best')
plt.grid(True)
plt.savefig('learning_curve_xgb.png')
plt.close()

# ---------------------------
# 4. Évaluation du modèle
# ---------------------------
print("\n" + "="*50)
print("ÉVALUATION DU MODÈLE")
print("="*50)

# Prédictions sur l'ensemble de test
y_pred = best_xgb.predict(X_test_selected)
y_prob = best_xgb.predict_proba(X_test_selected)[:, 1]

# Métriques de classification
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
avg_precision = average_precision_score(y_test, y_prob)

print(f"Précision: {accuracy:.4f}")
print(f"Score F1: {f1:.4f}")
print(f"Précision moyenne: {avg_precision:.4f}")

# Matrice de confusion
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Phishing', 'Phishing'],
            yticklabels=['Non-Phishing', 'Phishing'])
plt.xlabel('Prédiction')
plt.ylabel('Réalité')
plt.title('Matrice de confusion')
plt.savefig('confusion_matrix_xgb.png')
plt.close()

# Rapport de classification détaillé
print("\nRapport de classification:")
print(classification_report(y_test, y_pred))

# Courbe ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Courbe ROC (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs')
plt.title('Courbe ROC')
plt.legend(loc='lower right')
plt.savefig('roc_curve_xgb.png')
plt.close()

# Courbe Précision-Rappel
precision, recall, _ = precision_recall_curve(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', lw=2, label=f'Précision-Rappel (AP = {avg_precision:.4f})')
plt.axhline(y=sum(y_test)/len(y_test), color='red', linestyle='--', label='Référence')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Rappel')
plt.ylabel('Précision')
plt.title('Courbe Précision-Rappel')
plt.legend(loc='lower left')
plt.savefig('precision_recall_curve_xgb.png')
plt.close()

# ---------------------------
# 5. Enregistrement du modèle
# ---------------------------
print("\n" + "="*50)
print("ENREGISTREMENT DU MODÈLE")
print("="*50)

# Sauvegarder le modèle
best_xgb.save_model('xgboost_phishing_model.json')
print("Modèle enregistré sous 'xgboost_phishing_model.json'")

# Sauvegarde des transformations pour un usage ultérieur
import joblib
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(rfecv, 'feature_selector.pkl')
print("Transformations enregistrées (scaler.pkl, feature_selector.pkl)")

print("\n" + "="*80)
print("ANALYSE TERMINÉE")
print("="*80)

PHISHING DETECTION AVEC XGBOOST - ANALYSE AVANCÉE
Valeurs uniques initiales dans la colonne cible: [-1  1]
Valeurs après transformation: [0 1]
Dimensions du dataset: 11055 lignes × 31 colonnes

ANALYSE EXPLORATOIRE DES DONNÉES

Statistiques descriptives:
       having_IP_Address  URL_Length  Shortining_Service  having_At_Symbol  \
count         11055.0000  11055.0000          11055.0000        11055.0000   
mean              0.3138     -0.6332              0.7388            0.7006   
std               0.9495      0.7661              0.6740            0.7136   
min              -1.0000     -1.0000             -1.0000           -1.0000   
25%              -1.0000     -1.0000              1.0000            1.0000   
50%               1.0000     -1.0000              1.0000            1.0000   
75%               1.0000     -1.0000              1.0000            1.0000   
max               1.0000      1.0000              1.0000            1.0000   

       double_slash_redirecting  Prefix_Su

<Figure size 1200x800 with 0 Axes>